# Model Registry

Everything covered so far (log_model, evaluate, serve) dealt with a single model tied to a single run. In a real project you end up with dozens or hundreds of runs. Tracking "what's currently in production" via raw run IDs works but is fragile and manual.

The Model Registry solves this by turning logged models into a named, versioned, lifecycle-managed catalog.

## Model vs Registered Model

* Model = just logged to a run via `mlflow.log_model()`. Accessible only via `runs:/run_id/model`.

* Registered Model = that logged model has been registered under a name. It gets:
    * A version number (auto-incrementing, new registrations don't overwrite old ones)
    * Eligibility for a stage

## Stages

| Stage | Meaning |
|---|---|
| None | Registered, not yet evaluated for use |
| Staging | Under validation/testing |
| Production | Actively serving |
| Archived | Retired, kept but inactive |

# Registering a Model

### Creating a registered model
```
client.create_registered_model(name="MODEL_NAME")
```
* Each new record with the same name will result in INCREASED version number, as its automatically


### AUTOMATICALLY SAVING: Registering while logging
```
# During training run
mlflow.FLAVOR.log_model(name,
    artifact_uri,
    registered_model_name="MODEL_NAME")
```

Adding during log_model an parameter of registered_model_name is enough.

### SAVING AFTER YOU ARE SATISFIED WITH RESULTS: Registering from a explicit run id/ with model uri
```

# Existing MLflow Models
mlflow.register_model(model_uri, name)

# Register a model from a run
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="ames_housing_model"
)

```



# Loading a Model

```
# Search for registered models
client.search_registered_models(filter_string=MY_FILTER_STRING)
```
Also possible (and better) with looking with UI.

After finding the URI, loading is based on flavor. **mlflow.pyfunc** is generic, working with every flavor.

```
model = mlflow.pytorch.load_model("models:/ames_housing_full_pipeline/Production")


model = mlflow.pyfunc.load_model("models:/ames_housing_full_pipeline/Production")

```

## Loading current production/stating model or specific version


```
#Current production model:
model = mlflow.pyfunc.load_model("models:/ames_housing_full_pipeline/Production")
model = mlflow.pyfunc.load_model("models:/ames_housing_full_pipeline/Staging")


# Loading a specific model:
model = mlflow.pyfunc.load_model("models:/ames_housing_full_pipeline/3")

```

# Registering a Custom PyFunc model
Actually the method described here is applies to logging, as it first happens through logging and given registered_model_name.

Remember the python class written in 01_mlflow_models, to save models code aswell it must be saved in **a seperate python file**, not in the same notebook (Which is the optimal flow for an production system)

Also model and imputer used there is saved as artifacts (As created the class structure).

Then the python file can be passed as a parameter within `pyfunc.log_model`

In [ ]:
mlflow.pyfunc.log_model(
    artifact_path="ames_full_pipeline",
    python_model=CustomPredict(),
    artifacts={
        "imputer": f"runs:/{run_id}/imputer/imputer.pkl",
        "nn_model": f"runs:/{run_id}/nn_model"
    },
    code_path=["custom_predict.py"],  # python file path
    registered_model_name="ames_housing_full_pipeline" # registered model name
)

On the loading side, when you log the model with **code_path and then load it using load_model(),MLflow automatically adds the packaged .py files to sys.path and imports them as needed**.

So you do not need to manually write something like from custom_predict import CustomPredict at inference time.

* python_model tells MLflow which PythonModel implementation to use (for example, your CustomPredict() object or class instance that defines load_context() / predict()).
* code_path tells MLflow which source code files should be shipped with the model artifact so that the python_model class and any helper modules can actually be imported later in another environment.

In [ ]:

import mlflow.pyfunc

class CustomPredict(mlflow.pyfunc.PythonModel):

    def load_context(self, context):

        # context : it containts .artifacts

        #loads imputer function
        self.imputer = joblib.load(context.artifacts["imputer"])

        #loads any model from, we assume its saved under the artifact nn_model
        self.model = mlflow.pytorch.load_model(context.artifacts["nn_model"])

        # we switch to the evaluation mode at pytorch
        self.model.eval()

    def predict(self, context, model_input, params=None):


        # 1. Calling imputer on the model_input
        imputed = self.imputer.transform(model_input)

        # 2. PyTorch forward pass
        with torch.no_grad():
            tensor_input = torch.tensor(imputed, dtype=torch.float32)
            log_pred = self.model(tensor_input).numpy()

        # 3. Postprocessing
        return np.expm1(log_pred)


